In [ ]:
! pip install kaggle


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for unzip: filename=unzip-1.0.0-py3-none-any.whl size=1305 sha256=78520f4c7084f2d44851ab02ce4607aabcb16ae9f9f2592c66875faad32553f2
  Stored in directory: c:\users\fabia\appdata\local\pip\cache\wheels\fb\5b\81\0f3e1e533b52883f88ab978178c15627a4fce4c13f74911dce
Successfully built unzip


  DEPRECATION: Building 'unzip' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'unzip'. Discussion can be found at https://github.com/pypa/pip/issues/6334

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
! kaggle datasets download lokeshparab/amazon-products-dataset

Dataset URL: https://www.kaggle.com/datasets/lokeshparab/amazon-products-dataset
License(s): Attribution-NonCommercial 4.0 International (CC BY-NC 4.0)




  0%|          | 0.00/79.7M [00:00<?, ?B/s]
 79%|███████▉  | 63.0M/79.7M [00:00<00:00, 657MB/s]
100%|██████████| 79.7M/79.7M [00:00<00:00, 711MB/s]


Libraries for code

In [1]:
import zipfile, os
import pandas as pd
import numpy as np

We will only use the 'Amazon-Products.csv' file because it contains all the products.

In [2]:
zip_file = 'amazon-products-dataset.zip'
extract_dir = "datasetProducts"

if os.path.exists(zip_file):
    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print("Dataset extracted in", extract_dir)
else:
    print("Don't found zip file")

df = pd.read_csv('datasetProducts/Amazon-Products.csv')

print(f"dataset head:\n {df.head()}")
print(f"information:\n {df.info()}")
print(f"rows:\n {df.shape[0]}, columns:\n {df.shape[1]}")

Dataset extracted in datasetProducts
dataset head:
    Unnamed: 0                                               name  \
0           0  Lloyd 1.5 Ton 3 Star Inverter Split Ac (5 In 1...   
1           1  LG 1.5 Ton 5 Star AI DUAL Inverter Split AC (C...   
2           2  LG 1 Ton 4 Star Ai Dual Inverter Split Ac (Cop...   
3           3  LG 1.5 Ton 3 Star AI DUAL Inverter Split AC (C...   
4           4  Carrier 1.5 Ton 3 Star Inverter Split AC (Copp...   

  main_category      sub_category  \
0    appliances  Air Conditioners   
1    appliances  Air Conditioners   
2    appliances  Air Conditioners   
3    appliances  Air Conditioners   
4    appliances  Air Conditioners   

                                               image  \
0  https://m.media-amazon.com/images/I/31UISB90sY...   
1  https://m.media-amazon.com/images/I/51JFb7FctD...   
2  https://m.media-amazon.com/images/I/51JFb7FctD...   
3  https://m.media-amazon.com/images/I/51JFb7FctD...   
4  https://m.media-amazon.com/images

In [3]:
print(f"Columns Name:\n{df.columns.tolist()}")
print(f"Unique Values:\n{df.nunique()}")
print(f"Null values:\n {df.isnull().sum()}")

Columns Name:
['Unnamed: 0', 'name', 'main_category', 'sub_category', 'image', 'link', 'ratings', 'no_of_ratings', 'discount_price', 'actual_price']
Unique Values:
Unnamed: 0         19200
name              396210
main_category         20
sub_category         112
image             462414
link              551585
ratings               49
no_of_ratings       8342
discount_price     27511
actual_price       23170
dtype: int64
Null values:
 Unnamed: 0             0
name                   0
main_category          0
sub_category           0
image                  0
link                   0
ratings           175794
no_of_ratings     175794
discount_price     61163
actual_price       17813
dtype: int64


Now we're going to delete all the columns we won't be using for our products: image, link, no_of_ratings, and discount_price.

In [ ]:
df_modified = df.drop(columns=['image', 'link', 'no_of_ratings', 'discount_price']).copy()

# Clean ratings as text first
df_modified['ratings'] = df_modified['ratings'].astype(str).str.strip()

# Replace common strings meaning "no data" with NaN
df_modified['ratings'].replace({
    '', ' ', '—', '-', 'No ratings', 'Not available', 'NULL', 'None'
}, np.nan, inplace=True)

# If decimals use comma, replace with dot
df_modified['ratings'] = df_modified['ratings'].str.replace(',', '.', regex=False)

# Now convert to numeric (non-convertible -> NaN)
df_modified['ratings'] = pd.to_numeric(df_modified['ratings'], errors='coerce')

# Clean and convert price (you can keep the order you prefer)
df_modified['actual_price'] = (
    df_modified['actual_price']
    .replace('₹', '', regex=True)
    .replace(',', '', regex=True)
    .astype(float)
    .round(2)
)
df_modified['actual_price'] = df_modified['actual_price'] * 0.012

# Finally, drop rows without actual_price or ratings
df_modified = df_modified.dropna(subset=['actual_price', 'ratings'])

# Check result
print(df_modified.head())           # Note: use head() with parentheses
print("Nulls per column:\n", df_modified.isnull().sum())


C:\Users\fabia\AppData\Local\Temp\ipykernel_12420\1614560615.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_modified['ratings'].replace({


   Unnamed: 0                                               name  \
0           0  Lloyd 1.5 Ton 3 Star Inverter Split Ac (5 In 1...   
1           1  LG 1.5 Ton 5 Star AI DUAL Inverter Split AC (C...   
2           2  LG 1 Ton 4 Star Ai Dual Inverter Split Ac (Cop...   
3           3  LG 1.5 Ton 3 Star AI DUAL Inverter Split AC (C...   
4           4  Carrier 1.5 Ton 3 Star Inverter Split AC (Copp...   

  main_category      sub_category  ratings  actual_price  
0    appliances  Air Conditioners      4.2        707.88  
1    appliances  Air Conditioners      4.2        911.88  
2    appliances  Air Conditioners      4.2        743.88  
3    appliances  Air Conditioners      4.0        827.88  
4    appliances  Air Conditioners      4.1        813.48  
Nulos por columna:
 Unnamed: 0       0
name             0
main_category    0
sub_category     0
ratings          0
actual_price     0
dtype: int64


We need just 100 products, so this step is for choosing them

In [ ]:
n_samples = 100  # Number of products to sample

# Create bins for ratings and prices to ensure diversity
df_modified['_rating_bin'] = pd.qcut(df_modified['ratings'], q=5, duplicates='drop')
df_modified['_price_bin'] = pd.qcut(df_modified['actual_price'], q=5, duplicates='drop')

# Create a stratum combining main_category, sub_category, rating bin, and price bin
df_modified['_strata'] = (
    df_modified['main_category'] + '|' +
    df_modified['sub_category'] + '|' +
    df_modified['_rating_bin'].astype(str) + '|' +
    df_modified['_price_bin'].astype(str)
)

# Sampling: take 1 product from each stratum
sampled = df_modified.groupby('_strata', group_keys=False).apply(lambda x: x.sample(1, random_state=42))

# Adjust to have 100 products
if len(sampled) > n_samples:
    sampled = sampled.sample(n=n_samples, random_state=42)
elif len(sampled) < n_samples:
    remaining = df_modified.drop(sampled.index)
    sampled = pd.concat([sampled, remaining.sample(n=n_samples - len(sampled), random_state=42)])

# Reset index
sampled = sampled.reset_index(drop=True)

# Drop temporary columns used for sampling
sampled = sampled.drop(columns=['_rating_bin', '_price_bin', '_strata'])

print("Nulls:\n", sampled.isnull().sum())

Nulos por columna:
 Unnamed: 0       0
name             0
main_category    0
sub_category     0
ratings          0
actual_price     0
dtype: int64


C:\Users\fabia\AppData\Local\Temp\ipykernel_12420\955966853.py:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled = df_modified.groupby('_strata', group_keys=False).apply(lambda x: x.sample(1, random_state=42))


In [ ]:

# TIME ID NAME MAIN_CATEGORY SUB_CATEGORY PRECIO_INICIAL PRECIO_FINAL RATINGS AVAILABILITY
sampled['time'] = "00:00:00:000"
sampled['ID'] = range(1, len(sampled) + 1)
sampled['initialPrice'] = sampled['actual_price']
sampled['finalPrice'] = sampled['actual_price']
sampled['availability'] = 1

# Select columns in order
df_to_save = sampled[['time', 'ID', 'name', 'main_category', 'sub_category', 'initialPrice', 'finalPrice', 'ratings', 'availability']]
# Optional: replace spaces in names with underscores
df_to_save['name'] = df_to_save['name'].str.replace(' ', '_')
# Changes id's 1 to 100

# Normalizes text: replaces spaces with '_' and removes quotes
cols_to_fix = ['name', 'main_category', 'sub_category']

for col in cols_to_fix:
    df_to_save[col] = (
        df_to_save[col]
        .astype(str)                
        .str.replace("'", "", regex=False)   
        .str.replace('"', "", regex=False)  
        .str.replace(" ", "_", regex=False)  
    )

C:\Users\fabia\AppData\Local\Temp\ipykernel_12420\911829157.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_to_save['name'] = df_to_save['name'].str.replace(' ', '_')
C:\Users\fabia\AppData\Local\Temp\ipykernel_12420\911829157.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_to_save[col] = (
C:\Users\fabia\AppData\Local\Temp\ipykernel_12420\911829157.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = val

Finally, we save the file in the directory of cadmium

In [ ]:
output_file = "initial_product_information_test.txt"

# Destination directory: go up one level from dataset and enter input_data 
DIRECTORY_SAVE = os.path.join("..", "input_data") 
# Build the full path 
file_path = os.path.join(DIRECTORY_SAVE, output_file) 

# Save as txt, space-separated, without index or header 
df_to_save.to_csv(file_path, sep=' ', index=False, header=False) 
print(f"File saved in: {file_path}")


File saved in: ..\input_data\initial_product_information_test.txt
